# 01 · MVTec AD - Exploratory Data Analysis

**Arkon Manufacturing | Department: Component Inspection**

Dataset: Anomaly Detection - train only on GOOD images, then detect defects.

Categories in our subset: `grid`, `metal_nut`, `screw`, `transistor`

**Analogy:** Like a new QC inspector - first learn what a perfect part looks like, then the deviations stand out.

In [ ]:
import sys
from pathlib import Path

_nb_root = Path('../..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)
print('arkon_utils loaded ✓')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import pandas as pd
from collections import defaultdict

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
ASSETS = 'cv/mvtec'


In [ ]:
DATA_DIR   = Path('../../../data/05_mvtec/raw')
CATEGORIES = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f'Categories ({len(CATEGORIES)}): {CATEGORIES}')

## 1. Dataset Overview - structure per category

In [ ]:
rows = []
for cat in CATEGORIES:
    cat_dir = DATA_DIR / cat
    # Train: only good
    train_good = list((cat_dir / 'train' / 'good').glob('*.png'))
    # Test: good + defects
    test_dir = cat_dir / 'test'
    for defect_type in sorted(test_dir.iterdir()):
        if defect_type.is_dir():
            imgs = list(defect_type.glob('*.png'))
            rows.append({'category': cat, 'split': 'test',
                         'defect': defect_type.name, 'count': len(imgs)})
    rows.append({'category': cat, 'split': 'train', 'defect': 'good', 'count': len(train_good)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# Summary: train_good vs test_total per category
summary = df.groupby('category').apply(
    lambda g: pd.Series({
        'train_good': g[g['defect']=='good']['count'].sum(),
        'test_good':  g[(g['split']=='test') & (g['defect']=='good')]['count'].sum(),
        'test_defect': g[(g['split']=='test') & (g['defect']!='good')]['count'].sum()
    })
).reset_index()
print(summary.to_string(index=False))

## 2. Sample Images - Train (Good) vs Test (Defective)

In [ ]:
fig, axes = plt.subplots(len(CATEGORIES), 5, figsize=(16, 4*len(CATEGORIES)))
for row, cat in enumerate(CATEGORIES):
    cat_dir = DATA_DIR / cat
    # Col 0-1: train/good
    good_imgs = sorted((cat_dir/'train'/'good').glob('*.png'))[:2]
    for col, img_path in enumerate(good_imgs):
        axes[row][col].imshow(Image.open(img_path))
        axes[row][col].set_title('good', color='green', fontsize=8)
        axes[row][col].axis('off')
    # Col 2-4: test defects
    test_defects = [d for d in (cat_dir/'test').iterdir() if d.is_dir() and d.name != 'good']
    for col, defect_dir in zip(range(2, 5), test_defects):
        imgs = list(defect_dir.glob('*.png'))
        if imgs:
            axes[row][col].imshow(Image.open(imgs[0]))
            axes[row][col].set_title(defect_dir.name, color='red', fontsize=8)
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(cat, fontsize=10, fontweight='bold')
plt.suptitle('MVTec AD - Good vs Defective Samples', y=1.01)
plt.tight_layout()
save_figure(fig, 'cv_mvtec_plot_1', subfolder='cv/mvtec')
plt.show()

## 3. Ground Truth Masks - annotation visualisation

In [ ]:
# Show: image | defect | mask
cat = CATEGORIES[0]  # grid
cat_dir = DATA_DIR / cat
defect_types = [d for d in (cat_dir/'test').iterdir() if d.is_dir() and d.name != 'good']

if defect_types:
    defect_dir = defect_types[0]
    test_imgs = sorted(defect_dir.glob('*.png'))
    mask_dir  = cat_dir / 'ground_truth' / defect_dir.name

    fig, axes = plt.subplots(min(3, len(test_imgs)), 2, figsize=(8, 8))
    for i in range(min(3, len(test_imgs))):
        img  = Image.open(test_imgs[i]).convert('RGB')
        axes[i][0].imshow(img); axes[i][0].set_title('Defective'); axes[i][0].axis('off')
        mask_path = mask_dir / test_imgs[i].name.replace('.png', '_mask.png')
        if mask_path.exists():
            mask = Image.open(mask_path)
            axes[i][1].imshow(mask, cmap='hot'); axes[i][1].set_title('GT Mask'); axes[i][1].axis('off')
    plt.suptitle(f'MVTec - {cat}/{defect_dir.name} with Ground Truth Masks')
    plt.tight_layout(); save_figure(fig, 'cv_mvtec_plot_2', subfolder='cv/mvtec')
plt.show()

## Summary

| Parameter | Value |
|---|---|
| Task | Unsupervised Anomaly Detection |
| Train | Good images only |
| Test | Good + all defect types |
| GT Masks | Pixel-level defect annotation |
| Approach | Feature Extraction → Anomaly Score |

➡️ **Next step:** `02_mvtec_preprocessing.ipynb`